# 03 PySpark ML Fraud Modeling

This notebook builds supervised machine learning models in PySpark to predict fraudulent credit card transactions.

The modeling workflow includes:
- loading the transaction data with Spark
- applying shared cleaning and feature engineering
- training a Logistic Regression baseline
- comparing performance against a Random Forest model
- evaluating tradeoffs between predictive performance and operational speed

In [1]:
#Imports
from pathlib import Path

from pyspark.sql import functions as F

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql.window import Window
from pyspark.ml.functions import vector_to_array


from project.config import load_config
from project.spark_session import create_spark
from project.schemas import TRANSACTION_SCHEMA
from project.transform import prepare_transactions
from project.fraud_modeling import split_by_label, undersample_training, oversample_training, hybrid_sample_training, fit_and_score_model, summarize_binary_predictions, build_comparison_df, show_summary, run_experiment

## Load configuration and start Spark

We load the project configuration and start a local Spark session.  
This keeps the notebook aligned with the same reusable project structure used in the EDA and streaming notebooks.

In [2]:
repo_root = Path.cwd().parent
cfg = load_config(repo_root / "configs" / "local.yaml")

raw_csv = repo_root / cfg.paths.raw

spark = create_spark(cfg)
spark.sparkContext.setLogLevel("WARN")

print("Spark session created.")
print(f"Raw data path: {raw_csv}")

Spark session created.
Raw data path: /home/jovyan/work/data/raw/credit_card_transactions.csv


## Read the dataset with Spark

We load the full transaction dataset using the shared transaction schema.  
The same schema is reused across the project for consistency.

In [3]:
df = (spark.read
         .option("header", True)
         .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
         .option("dateFormat", "yyyy-MM-dd")
         .schema(TRANSACTION_SCHEMA)
         .csv(str(raw_csv))
)

In [4]:
df.printSchema()
print(f"Row count: {df.count():,}")

root
 |-- Unnamed: 0: long (nullable = true)
 |-- trans_date_trans_time: timestamp (nullable = true)
 |-- cc_num: string (nullable = true)
 |-- merchant: string (nullable = true)
 |-- category: string (nullable = true)
 |-- amt: double (nullable = true)
 |-- first: string (nullable = true)
 |-- last: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- street: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- city_pop: integer (nullable = true)
 |-- job: string (nullable = true)
 |-- dob: date (nullable = true)
 |-- trans_num: string (nullable = true)
 |-- unix_time: long (nullable = true)
 |-- merch_lat: double (nullable = true)
 |-- merch_long: double (nullable = true)
 |-- is_fraud: integer (nullable = true)
 |-- merch_zipcode: string (nullable = true)

Row count: 1,296,675


## Apply reusable data transformations

We apply the shared transformation pipeline from `transform.py` to clean and enrich the dataset.

This step adds modeling-friendly features such as:
- transaction hour
- day of week
- customer age
- log-transformed transaction amount

In [5]:
df = prepare_transactions(df)

In [6]:
df.select(
    "amt",
    "category",
    "gender",
    "state",
    "event_hour",
    "event_dayofweek",
    "age",
    "is_fraud"
).show(10, truncate=False)

+------+-------------+------+-----+----------+---------------+---+--------+
|amt   |category     |gender|state|event_hour|event_dayofweek|age|is_fraud|
+------+-------------+------+-----+----------+---------------+---+--------+
|4.97  |misc_net     |F     |NC   |0         |3              |38 |0       |
|107.23|grocery_pos  |F     |WA   |0         |3              |47 |0       |
|220.11|entertainment|M     |ID   |0         |3              |64 |0       |
|45.0  |gas_transport|M     |MT   |0         |3              |59 |0       |
|41.96 |misc_pos     |M     |VA   |0         |3              |40 |0       |
|94.63 |gas_transport|F     |PA   |0         |3              |64 |0       |
|44.54 |grocery_net  |F     |KS   |0         |3              |32 |0       |
|71.65 |gas_transport|M     |VA   |0         |3              |78 |0       |
|4.27  |misc_pos     |F     |PA   |0         |3              |85 |0       |
|198.39|grocery_pos  |F     |TN   |0         |3              |52 |0       |
+------+----

## Select modeling features

For the first-pass fraud model, we use a focused set of numeric and categorical features that are likely to be predictive while remaining efficient and interpretable.

We avoid identifier columns such as account numbers and transaction IDs in the initial model.

In [7]:
numeric_features = [
    "amt",
    "city_pop",
    "event_hour",
    "event_dayofweek",
    "age",
]

categorical_features = [
    "category",
    "gender",
    "state",
]

label_col = "is_fraud"

In [8]:
model_cols = numeric_features + categorical_features + [label_col]

model_df = df.select(*model_cols).dropna()

print(f"Modeling row count after dropna: {model_df.count():,}")
model_df.groupBy("is_fraud").count().show()

Modeling row count after dropna: 1,296,675
+--------+-------+
|is_fraud|  count|
+--------+-------+
|       1|   7506|
|       0|1289169|
+--------+-------+



## Split the data into training and test sets

We create a train/test split so that model performance can be evaluated on unseen transactions.

In [9]:
train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=5420)

print(f"Train rows: {train_df.count():,}")
print(f"Test rows: {test_df.count():,}")

Train rows: 1,036,349
Test rows: 260,326


## Build the PySpark feature pipeline

We encode categorical variables, assemble numeric and encoded categorical features into a single feature vector, and pass that vector to the classifier.

In [10]:
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_features
]

encoders = [
    OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe")
    for c in categorical_features
]

assembler_inputs = numeric_features + [f"{c}_ohe" for c in categorical_features]

assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features"
)

## Train a Logistic Regression baseline

Logistic Regression provides a strong baseline because it is fast, scalable, and relatively interpretable for binary classification problems such as fraud detection.

In [11]:
lr = LogisticRegression(
    featuresCol="features",
    labelCol=label_col,
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxIter=50,
    regParam=0.0,
    elasticNetParam=0.0
)

lr_pipeline = Pipeline(stages=indexers + encoders + [assembler, lr])

In [12]:
lr_model = lr_pipeline.fit(train_df)
lr_predictions = lr_model.transform(test_df)

In [13]:
lr_predictions.select(
    "is_fraud",
    "prediction",
    "probability"
).show(10, truncate=False)

+--------+----------+------------------------------------------+
|is_fraud|prediction|probability                               |
+--------+----------+------------------------------------------+
|0       |0.0       |[0.9989338458856225,0.0010661541143774933]|
|0       |0.0       |[0.9990080663088323,9.91933691167679E-4]  |
|0       |0.0       |[0.9995611451828336,4.3885481716643415E-4]|
|0       |0.0       |[0.999995156592025,4.843407975019254E-6]  |
|0       |0.0       |[0.9999915426989545,8.45730104548359E-6]  |
|0       |0.0       |[0.9962471886018401,0.003752811398159861] |
|0       |0.0       |[0.9986779973623716,0.0013220026376283833]|
|0       |0.0       |[0.9967358890335511,0.003264110966448852] |
|0       |0.0       |[0.9983928839960582,0.0016071160039418109]|
|0       |0.0       |[0.9999968330847192,3.166915280794491E-6] |
+--------+----------+------------------------------------------+
only showing top 10 rows


## Evaluate Logistic Regression

Because fraud detection is an imbalanced classification problem, accuracy alone is not sufficient.  
We evaluate the model using AUC, precision, recall, and F1.

In [14]:
binary_eval = BinaryClassificationEvaluator(
    labelCol=label_col,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

f1_eval = MulticlassClassificationEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="f1"
)

precision_eval = MulticlassClassificationEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="weightedPrecision"
)

recall_eval = MulticlassClassificationEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="weightedRecall"
)

In [15]:
lr_auc = binary_eval.evaluate(lr_predictions)
lr_f1 = f1_eval.evaluate(lr_predictions)
lr_precision = precision_eval.evaluate(lr_predictions)
lr_recall = recall_eval.evaluate(lr_predictions)

print("Logistic Regression Metrics")
print(f"AUC:       {lr_auc:.4f}")
print(f"F1:        {lr_f1:.4f}")
print(f"Precision: {lr_precision:.4f}")
print(f"Recall:    {lr_recall:.4f}")

Logistic Regression Metrics
AUC:       0.8198
F1:        0.9911
Precision: 0.9892
Recall:    0.9937


In [16]:
lr_predictions.groupBy("is_fraud", "prediction").count().orderBy("is_fraud", "prediction").show()

+--------+----------+------+
|is_fraud|prediction| count|
+--------+----------+------+
|       0|       0.0|258676|
|       0|       1.0|   115|
|       1|       0.0|  1515|
|       1|       1.0|    20|
+--------+----------+------+



## Train a Random Forest comparison model

Random Forest can capture nonlinear patterns and interactions that Logistic Regression may miss.  
We compare its predictive performance against the faster, simpler baseline.

In [17]:
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol=label_col,
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    numTrees=100,
    maxDepth=10,
    seed=5420
)

rf_pipeline = Pipeline(stages=indexers + encoders + [assembler, rf])

In [18]:
rf_model = rf_pipeline.fit(train_df)
rf_predictions = rf_model.transform(test_df)

In [19]:
rf_auc = binary_eval.evaluate(rf_predictions)
rf_f1 = f1_eval.evaluate(rf_predictions)
rf_precision = precision_eval.evaluate(rf_predictions)
rf_recall = recall_eval.evaluate(rf_predictions)

print("Random Forest Metrics")
print(f"AUC:       {rf_auc:.4f}")
print(f"F1:        {rf_f1:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall:    {rf_recall:.4f}")

Random Forest Metrics
AUC:       0.9829
F1:        0.9943
Precision: 0.9956
Recall:    0.9956


In [20]:
rf_predictions.groupBy("is_fraud", "prediction").count().orderBy("is_fraud", "prediction").show()

+--------+----------+------+
|is_fraud|prediction| count|
+--------+----------+------+
|       0|       0.0|258785|
|       0|       1.0|     6|
|       1|       0.0|  1135|
|       1|       1.0|   400|
+--------+----------+------+



## Compare model performance

We compare the two models to assess the tradeoff between:
- predictive power
- model complexity
- operational suitability for near-real-time fraud scoring

In [21]:
comparison_rows = [
    ("Logistic Regression", round(lr_auc, 6), round(lr_f1, 6), round(lr_precision, 6), round(lr_recall, 6)),
    ("Random Forest", round(rf_auc, 6), round(rf_f1, 6), round(rf_precision, 6), round(rf_recall, 6)),
]

comparison_df = spark.createDataFrame(
    comparison_rows,
    ["model", "auc", "f1", "precision", "recall"]
)

comparison_df.show()

+-------------------+--------+--------+---------+--------+
|              model|     auc|      f1|precision|  recall|
+-------------------+--------+--------+---------+--------+
|Logistic Regression|0.819763|0.991123| 0.989189|0.993739|
|      Random Forest|0.982916|0.994347| 0.995572|0.995617|
+-------------------+--------+--------+---------+--------+



## Model Performance Observation

At first glance, both Logistic Regression and Random Forest models appear to perform extremely well, with high AUC, F1, precision, and recall scores.

However, these metrics are misleading due to the extreme class imbalance in the dataset.

Fraudulent transactions represent only a small fraction of the data. As a result, the models can achieve high overall performance by simply predicting the majority class (non-fraud) for most observations.

This behavior is confirmed by examining prediction outputs and confusion matrices, where the model rarely identifies fraudulent transactions correctly.

Key issue:
- The reported F1, precision, and recall are **weighted metrics**, which are dominated by the majority class (non-fraud)
- As a result, strong performance on non-fraud inflates the overall metrics
- The model still performs poorly at detecting fraud, which is the primary objective

This highlights a fundamental challenge in fraud detection:
- Standard evaluation metrics can be misleading under class imbalance
- Models must be specifically optimized to detect minority class behavior

To address this, the next steps include:
- applying class weighting to emphasize fraud cases
- engineering features that better capture anomalous behavior
- re-evaluating model performance with improved techniques

In [22]:
window = Window.partitionBy()

print("Logistic Regression – Fraud-only performance")

lr_fraud = lr_predictions.filter(F.col("is_fraud") == 1)

lr_fraud_summary = (
    lr_fraud
    .groupBy("prediction")
    .count()
    .withColumn("total_fraud", F.sum("count").over(window))
    .withColumn("percentage", F.col("count") / F.col("total_fraud"))
    .withColumn("percentage", F.round(F.col("percentage") * 100, 2))
    .orderBy("prediction")
)

lr_fraud_summary.show()

Logistic Regression – Fraud-only performance
+----------+-----+-----------+----------+
|prediction|count|total_fraud|percentage|
+----------+-----+-----------+----------+
|       0.0| 1515|       1535|      98.7|
|       1.0|   20|       1535|       1.3|
+----------+-----+-----------+----------+



In [23]:
print("Random Forest – Fraud-only performance")

rf_fraud = rf_predictions.filter(F.col("is_fraud") == 1)

rf_fraud_summary = (
    rf_fraud
    .groupBy("prediction")
    .count()
    .withColumn("total_fraud", F.sum("count").over(window))
    .withColumn("percentage", F.col("count") / F.col("total_fraud"))
    .withColumn("percentage", F.round(F.col("percentage") * 100, 2))
    .orderBy("prediction")
)

rf_fraud_summary.show()

Random Forest – Fraud-only performance
+----------+-----+-----------+----------+
|prediction|count|total_fraud|percentage|
+----------+-----+-----------+----------+
|       0.0| 1135|       1535|     73.94|
|       1.0|  400|       1535|     26.06|
+----------+-----+-----------+----------+



### Fraud Detection Performance

The fraud-only evaluation highlights a critical issue with both models.

- Logistic Regression correctly identifies only **~2%** of fraud cases
- Random Forest improves performance but still captures only **~33%** of fraud cases

This means that the majority of fraudulent transactions are being classified as non-fraud.

Despite strong overall evaluation metrics, both models perform poorly on the actual objective of fraud detection due to class imbalance. The models are biased toward predicting the majority class (non-fraud), which dominates the dataset.

### Next Steps: Improving Fraud Detection

To address these limitations, we will improve the modeling approach using the following strategies:

1. **Class Imbalance Handling**
   - Oversampling/Undersampling
   - Apply class weighting to penalize misclassification of fraud cases more heavily

2. **Feature Engineering**
   - Introduce features that better capture anomalous behavior, such as:
     - transactions occurring at unusual times (e.g., late night)
     - abnormal transaction amounts
     - geographic distance between customer and merchant

3. **Model Re-evaluation**
   - Retrain models using the improved features and weighting
   - Compare fraud detection performance before and after improvements

These steps aim to increase the model’s ability to detect fraudulent transactions while maintaining scalability for real-world applications.

### Modeling note

At this stage, imbalance experiments are being tested with Logistic Regression only. This allows faster iteration and clearer comparison across resampling strategies.

Because Random Forest is more computationally expensive, it will be reintroduced later once the most effective class-balancing approach has been identified.

## Imbalance Handling

The baseline models performed poorly for fraud detection because the dataset is highly imbalanced. Most transactions are non-fraud, so the models tend to favor the majority class.

To address this, the following resampling strategies are tested:

1. **Undersampling**  
   Reduce the number of non-fraud transactions in the training set.

2. **Oversampling**  
   Duplicate fraud transactions in the training set.

3. **Hybrid sampling**  
   Combine limited oversampling of fraud with undersampling of non-fraud.

Important:
- Resampling is applied **only to the training data**
- The **test set remains untouched**
- This ensures model evaluation reflects performance on real-world class imbalance rather than an artificially balanced test set

In [24]:
fraud_train_df, nonfraud_train_df = split_by_label(train_df, label_col="is_fraud")

fraud_train_count = fraud_train_df.count()
nonfraud_train_count = nonfraud_train_df.count()

print(f"Fraud rows in training set: {fraud_train_count:,}\nNon-fraud rows in training set: {nonfraud_train_count:,}")

Fraud rows in training set: 5,971
Non-fraud rows in training set: 1,030,378


### Why resample only the training set?

If oversampling or undersampling is applied before the train/test split, duplicated or reduced examples may distort evaluation results.

By keeping the test set unchanged, all models are evaluated on the same original class distribution. This provides a more realistic comparison of fraud detection performance.

### Undersampling

Undersampling reduces the majority class (non-fraud) to better match the minority class (fraud).

This helps the model focus on fraud patterns, but it may discard useful information from valid transactions.

In [25]:
under_train_balanced_df = undersample_training(
    fraud_train_df,
    nonfraud_train_df,
    label_col="is_fraud",
    seed=5420
)

print(f"Undersampled training rows: {under_train_balanced_df.count():,}")
under_train_balanced_df.groupBy("is_fraud").count().show()

under_lr_predictions, under_lr_summary = run_experiment(
    lr_pipeline,
    under_train_balanced_df,
    test_df,
    model_name="Undersampled Logistic Regression",
    label_col="is_fraud",
    pred_col="prediction"
)

show_summary(spark, under_lr_summary)

Undersampled training rows: 11,982
+--------+-----+
|is_fraud|count|
+--------+-----+
|       1| 5971|
|       0| 6011|
+--------+-----+

+--------------------------------+------+-----+---+----+---------------+------------+--------+---------------+
|model                           |tn    |fp   |fn |tp  |fraud_precision|fraud_recall|fraud_f1|nonfraud_recall|
+--------------------------------+------+-----+---+----+---------------+------------+--------+---------------+
|Undersampled Logistic Regression|230117|28674|396|1139|0.0382         |0.742       |0.0727  |0.8892         |
+--------------------------------+------+-----+---+----+---------------+------------+--------+---------------+



#### Undersampling Results

With undersampling, our Logistic Regression model, undergoing no parameter changes, was able to increase its accuracy to 76%. A substantial increase from our initial outcome of ~1%.

### Oversampling

Oversampling increases the number of fraud transactions by duplicating minority-class examples.

This preserves all non-fraud data, but repeated fraud examples may increase the risk of overfitting.

In [26]:
over_train_balanced_df = oversample_training(
    fraud_train_df,
    nonfraud_train_df,
    label_col="is_fraud",
    seed=5420
)

print(f"Oversampled training rows: {over_train_balanced_df.count():,}")
over_train_balanced_df.groupBy("is_fraud").count().show()

over_lr_predictions, over_lr_summary = run_experiment(
    lr_pipeline,
    over_train_balanced_df,
    test_df,
    model_name="Oversampled Logistic Regression",
    label_col="is_fraud",
    pred_col="prediction"
)

build_comparison_df(spark, [over_lr_summary]).show(truncate=False)

Oversampled training rows: 2,060,252
+--------+-------+
|is_fraud|  count|
+--------+-------+
|       0|1030378|
|       1|1029874|
+--------+-------+

+-------------------------------+------+-----+---+----+---------------+------------+--------+---------------+
|model                          |tn    |fp   |fn |tp  |fraud_precision|fraud_recall|fraud_f1|nonfraud_recall|
+-------------------------------+------+-----+---+----+---------------+------------+--------+---------------+
|Oversampled Logistic Regression|229609|29182|397|1138|0.0375         |0.7414      |0.0714  |0.8872         |
+-------------------------------+------+-----+---+----+---------------+------------+--------+---------------+



#### Oversampling Results

With oversampling, our Logistic Regression model increased fraud recall to 76%, a substantial improvement over the baseline fraud recall of about 1%.

### Imbalance Summary

Based on the outcomes, undersampling and oversampling led to roughly same outcome of ~75% fraud recall on fraudulent charges. Undersampling, being only ~15,000 rows of data, was able to produce an outcome much quicker and with similiar results with less data. With oversampling, there is much more data to evaluate, but with the fraudulent data needing to be duplicated over 150 times, this very likely would lead to overfitting once we tweek our hyper parameters on our models since it would recognize the same datapoints reproduced that many times. 

### Imbalance Conclusion

The initial resampling experiments suggest that a hybrid approach may provide a better balance between information retention and class balance than pure undersampling or pure oversampling alone.

Undersampling performed well with much lower computational cost, while oversampling preserved majority-class data but required substantial duplication of fraud observations. Hybrid sampling is intended to balance these tradeoffs and should be evaluated using fraud recall, fraud precision, and fraud F1 on the unchanged test set.

### Hybrid Sampling

Hybrid sampling combines limited oversampling of fraud with undersampling of non-fraud.

This approach aims to:
- avoid discarding too much majority-class data
- avoid excessive duplication of minority-class data
- create a more balanced training set using a middle-ground strategy

In [27]:
hybrid_train_df = hybrid_sample_training(
    fraud_train_df,
    nonfraud_train_df,
    fraud_multiplier=5.0,
    seed=5420
)

print(f"Hybrid training rows: {hybrid_train_df.count():,}")
hybrid_train_df.groupBy("is_fraud").count().show()

hybrid_lr_predictions, hybrid_lr_summary = run_experiment(
    lr_pipeline,
    hybrid_train_df,
    test_df,
    model_name="Hybrid Logistic Regression",
    label_col="is_fraud",
    pred_col="prediction"
)

build_comparison_df(spark, [hybrid_lr_summary]).show(truncate=False)

Hybrid training rows: 59,710
+--------+-----+
|is_fraud|count|
+--------+-----+
|       1|29855|
|       0|29855|
+--------+-----+

+--------------------------+------+-----+---+----+---------------+------------+--------+---------------+
|model                     |tn    |fp   |fn |tp  |fraud_precision|fraud_recall|fraud_f1|nonfraud_recall|
+--------------------------+------+-----+---+----+---------------+------------+--------+---------------+
|Hybrid Logistic Regression|230132|28659|395|1140|0.0383         |0.7427      |0.0728  |0.8893         |
+--------------------------+------+-----+---+----+---------------+------------+--------+---------------+



#### Hybrid Sampling Results

Hybrid sampling combines moderate fraud oversampling with non-fraud undersampling to create a more balanced training set without fully discarding majority-class information or excessively duplicating minority-class examples.

This method should be compared against the undersampling and oversampling results using fraud recall, fraud precision, and fraud F1 on the unchanged test set.

In [28]:
baseline_lr_summary = run_experiment(
    lr_pipeline,
    train_df,
    test_df,
    model_name="Baseline Logistic Regression",
    label_col="is_fraud",
    pred_col="prediction"
)[1]

comparison_df = build_comparison_df(
    spark,
    [
        baseline_lr_summary,
        under_lr_summary,
        over_lr_summary,
        hybrid_lr_summary,
    ]
)

comparison_df.show(truncate=False)

+--------------------------------+------+-----+----+----+---------------+------------+--------+---------------+
|model                           |tn    |fp   |fn  |tp  |fraud_precision|fraud_recall|fraud_f1|nonfraud_recall|
+--------------------------------+------+-----+----+----+---------------+------------+--------+---------------+
|Baseline Logistic Regression    |258676|115  |1515|20  |0.1481         |0.013       |0.024   |0.9996         |
|Undersampled Logistic Regression|230117|28674|396 |1139|0.0382         |0.742       |0.0727  |0.8892         |
|Oversampled Logistic Regression |229609|29182|397 |1138|0.0375         |0.7414      |0.0714  |0.8872         |
|Hybrid Logistic Regression      |230132|28659|395 |1140|0.0383         |0.7427      |0.0728  |0.8893         |
+--------------------------------+------+-----+----+----+---------------+------------+--------+---------------+



Conclusion: With all methods performing roughy the same. It becomes a choice between the Hybrid method and the Undersample method.

Oversampling takes too long to run and operationally would be far too expensive. Hybrid is likely better since it allows for an "artificially" larger dataset with duplicated some fraud data without going overboard and pulling in additional data from the true legit credit card transactions.

Next Steps:

## Class Weights

Another method to address the imbalance, class weights will be tested. Rather than changing the class distribution through undersampling or oversampling, weighting changes the training loss so that errors on fraudulent transactions carry a larger penalty than errors on non-fraudulent ones.

In practice, this means:

- the fraud class receives a higher weight/penalty
- the non-fraud class receives a lower weight/penalty
- the model is encouraged to pay more attention to the minority class

This approach helps improve fraud detection while preserving the full training dataset. Compared with resampling, class weighting avoids removing legitimate transactions or duplicating fraudulent ones, making it a useful alternative for handling imbalance in an interpretable way.

In [29]:
# Calculating weights (using the train df)
total_count = train_df.count()

fraud_count = train_df.filter(F.col("is_fraud") == 1).count()
nonfraud_count = train_df.filter(F.col("is_fraud") == 0).count()

num_classes = 2

weight_0 = total_count / (num_classes * nonfraud_count) #majority
weight_1 = total_count / (num_classes * fraud_count) #minority

In [30]:
print(f"With the weights now in place;\n\tNon-fraud data is weighted as {round(weight_0,4)}\n\tFraud data is weighted as {round(weight_1,4)}\nWith these weights in place, the contributions are now equalized without altering our actual data\n\tNon-fraud contribution: {nonfraud_count * weight_0}\n\tFraud contribution: {fraud_count * weight_1}" )

With the weights now in place;
	Non-fraud data is weighted as 0.5029
	Fraud data is weighted as 86.7819
With these weights in place, the contributions are now equalized without altering our actual data
	Non-fraud contribution: 518174.49999999994
	Fraud contribution: 518174.5


In [31]:
train_weighted = train_df.withColumn(
    "class_weight",
    F.when(F.col("is_fraud") == 1, weight_1).otherwise(weight_0)
)

test_weighted = test_df.withColumn("class_weight", F.lit(1.0))

In [32]:
# Slight alteration to our logistic regression model so it can use weights
lr_w = LogisticRegression(
    featuresCol="features",
    weightCol="class_weight", # This is the only change, all other parameters preserved
    labelCol=label_col,
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxIter=50,
    regParam=0.0,
    elasticNetParam=0.0
)

lr_w_pipeline = Pipeline(stages=indexers + encoders + [assembler, lr_w])

In [33]:
weighted_preds, weighted_summary = run_experiment(
    lr_w_pipeline,
    train_weighted,
    test_weighted,
    model_name="Weighted Logistic Regression",
    label_col="is_fraud",
    pred_col="prediction"
)

In [34]:
build_comparison_df(spark, [weighted_summary]).show(truncate=False)

+----------------------------+------+-----+---+----+---------------+------------+--------+---------------+
|model                       |tn    |fp   |fn |tp  |fraud_precision|fraud_recall|fraud_f1|nonfraud_recall|
+----------------------------+------+-----+---+----+---------------+------------+--------+---------------+
|Weighted Logistic Regression|229519|29272|398|1137|0.0374         |0.7407      |0.0712  |0.8869         |
+----------------------------+------+-----+---+----+---------------+------------+--------+---------------+



### Weighted Results

The results are very similar to our earlier methods of bringing out dataset in to balance. We will compare all methods outcomes below and decide on how to proceed.

## Imbalance Correction Results

In [35]:
comparison_df = build_comparison_df(
    spark,
    [
        baseline_lr_summary,
        under_lr_summary,
        over_lr_summary,
        hybrid_lr_summary,
        weighted_summary
    ]
)

comparison_df.show(truncate=False)

+--------------------------------+------+-----+----+----+---------------+------------+--------+---------------+
|model                           |tn    |fp   |fn  |tp  |fraud_precision|fraud_recall|fraud_f1|nonfraud_recall|
+--------------------------------+------+-----+----+----+---------------+------------+--------+---------------+
|Baseline Logistic Regression    |258676|115  |1515|20  |0.1481         |0.013       |0.024   |0.9996         |
|Undersampled Logistic Regression|230117|28674|396 |1139|0.0382         |0.742       |0.0727  |0.8892         |
|Oversampled Logistic Regression |229609|29182|397 |1138|0.0375         |0.7414      |0.0714  |0.8872         |
|Hybrid Logistic Regression      |230132|28659|395 |1140|0.0383         |0.7427      |0.0728  |0.8893         |
|Weighted Logistic Regression    |229519|29272|398 |1137|0.0374         |0.7407      |0.0712  |0.8869         |
+--------------------------------+------+-----+----+----+---------------+------------+--------+---------

Given that all imbalance-handling methods produced similar performance, the preferred approach is the one that minimizes computational cost while maintaining effectiveness. 

In this case, undersampling and weighted logistic regression provide comparable results to more computationally intensive methods, making them more practical choices for further modeling and potential real-world deployment.

## Feature Engineering

The baseline and imbalance-adjusted models improved fraud detection performance, but results plateaued across all approaches. This suggests that model performance is now limited by the available features rather than the modeling technique.

To address this, additional features are engineered to better capture anomalous transaction behavior.

These features focus on:

- Temporal patterns (e.g., transactions at unusual times)
- Transaction characteristics (e.g., unusually large amounts)
- Geographic behavior (e.g., distance between customer and merchant)
- Behavioral patterns (e.g., transaction frequency)

The goal is to provide the model with more context around each transaction, allowing it to better distinguish between normal and potentially fraudulent activity.

### Transaction Time

In [36]:
feat_eng_df = (
    df
    .withColumn("is_night", F.when(F.col("event_hour").between(0, 5), 1).otherwise(0))
    .withColumn("is_weekend", F.when(F.col("event_dayofweek").isin([1,7]), 1).otherwise(0))
)

### Amount Behavior

In [37]:
feat_eng_df = feat_eng_df.withColumn("amt_log", F.log1p("amt"))

feat_eng_df = feat_eng_df.withColumn(
    "high_amt_flag",
    F.when(F.col("amt") > 200, 1).otherwise(0)
)

### Geographic Distance

In [38]:
feat_eng_df = feat_eng_df.withColumn(
    "lat_diff",
    F.abs(F.col("lat") - F.col("merch_lat"))
).withColumn(
    "long_diff",
    F.abs(F.col("long") - F.col("merch_long"))
)

feat_eng_df = feat_eng_df.withColumn(
    "distance_proxy",
    F.sqrt(F.col("lat_diff")**2 + F.col("long_diff")**2)
)

### Transaction Frequency

**Note**

While card-level behavioral features (e.g., average transaction amount per card) can provide strong predictive signals, they must be used carefully to avoid data leakage. This dataset has already hashed or anonymized the card details, making this safe for us to utilize.

In a production system, such features would be computed using only historical transactions available at the time of scoring. For this analysis, simplified aggregates are used to approximate behavioral patterns.

In [39]:
window_cc = Window.partitionBy("cc_num")

feat_eng_df = feat_eng_df.withColumn(
    "avg_amt_per_card",
    F.avg("amt").over(window_cc)
)

feat_eng_df = feat_eng_df.withColumn(
    "amt_vs_avg",
    F.col("amt") / F.col("avg_amt_per_card")
)

In [40]:
feat_eng_df.select(
    F.mean("is_night").alias("pct_night"),
    F.mean("is_weekend").alias("pct_weekend"),
    F.mean("high_amt_flag").alias("pct_high_amt"),
    F.avg("distance_proxy").alias("avg_distance_proxy"),
    F.avg("avg_amt_per_card").alias("avg_amt_per_card"),
    F.avg("amt_vs_avg").alias("avg_amt_vs_avg")
).show()

+------------------+-------------------+-------------------+------------------+-----------------+------------------+
|         pct_night|        pct_weekend|       pct_high_amt|avg_distance_proxy| avg_amt_per_card|    avg_amt_vs_avg|
+------------------+-------------------+-------------------+------------------+-----------------+------------------+
|0.1965257292690921|0.34822603967840826|0.04775599128540305|0.7656610714225209|70.35103545607116|0.9999999999999933|
+------------------+-------------------+-------------------+------------------+-----------------+------------------+



In [41]:
feat_eng_df.groupBy("is_fraud").agg(
    F.avg("amt").alias("avg_amt"),
    F.avg("amt_log").alias("avg_amt_log"),
    F.avg("is_night").alias("pct_night"),
    F.avg("is_weekend").alias("pct_weekend"),
    F.avg("high_amt_flag").alias("pct_high_amt"),
    F.avg("distance_proxy").alias("avg_distance_proxy"),
    F.avg("amt_vs_avg").alias("avg_amt_vs_avg")
).show(truncate=False)

+--------+-----------------+------------------+-------------------+-------------------+------------------+------------------+------------------+
|is_fraud|avg_amt          |avg_amt_log       |pct_night          |pct_weekend        |pct_high_amt      |avg_distance_proxy|avg_amt_vs_avg    |
+--------+-----------------+------------------+-------------------+-------------------+------------------+------------------+------------------+
|1       |531.3200919264585|5.568740632416058 |0.35078603783639756|0.32547295496935785|0.7599253930189181|0.7672812843648927|6.912082374216447 |
|0       |67.66710981259993|3.5216203669255615|0.19562757093910885|0.3483585162224658 |0.0436094879724846|0.7656516379670585|0.9655777556698326|
+--------+-----------------+------------------+-------------------+-------------------+------------------+------------------+------------------+



In [42]:
feat_eng_df.filter(F.col("is_fraud") == 1).select(
    "cc_num",
    "amt",
    "avg_amt_per_card",
    "amt_vs_avg"
).orderBy(F.desc("amt_vs_avg")).show(10, truncate=False)

+----------------+-------+------------------+------------------+
|cc_num          |amt    |avg_amt_per_card  |amt_vs_avg        |
+----------------+-------+------------------+------------------+
|2720433095629877|1262.19|52.78476343739943 |23.91201395639304 |
|3576021480694169|1186.46|50.29796992481205 |23.58862597781939 |
|30561214688470  |1131.32|48.48821547223542 |23.331854739999642|
|38052002992326  |1118.61|47.94510046367857 |23.331059674124937|
|3533742182628021|1086.16|46.98265945110159 |23.11831668725456 |
|6534628260579800|1202.36|52.44669344473002 |22.925372812436397|
|3518669219150142|1182.65|51.635493069306925|22.903819247210574|
|4364010865167176|1076.69|47.18054799070068 |22.82063362664241 |
|4364010865167176|1074.41|47.18054799070068 |22.772308626253494|
|2296006538441789|1132.3 |50.01097393015256 |22.641030778193162|
+----------------+-------+------------------+------------------+
only showing top 10 rows


In [43]:
numeric_features = [
    #"amt", # Similar to "amt_log", but weaker
    "city_pop",
    "event_hour",
    "event_dayofweek",
    "age",
    "amt_log",
    #"high_amt_flag", # Similar to "amt_log", but weaker
    "is_night",
    "amt_vs_avg",
]

categorical_features = [
    "category",
    "gender",
    "state",
]

label_col = "is_fraud"

In [44]:
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_features
]

encoders = [
    OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe")
    for c in categorical_features
]

assembler_inputs = numeric_features + [f"{c}_ohe" for c in categorical_features]

assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features"
)

In [45]:
model_cols = numeric_features + categorical_features + [label_col]
model_df = feat_eng_df.select(*model_cols).dropna()

print(f"Feature-engineered modeling row count: {model_df.count():,}")
model_df.groupBy("is_fraud").count().show()

Feature-engineered modeling row count: 1,296,675
+--------+-------+
|is_fraud|  count|
+--------+-------+
|       1|   7506|
|       0|1289169|
+--------+-------+



In [46]:
train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=5420)

print(f"Train rows: {train_df.count():,}")
print(f"Test rows: {test_df.count():,}")

Train rows: 1,036,767
Test rows: 259,908


In [ ]:
total_count = train_df.count()
fraud_count = train_df.filter(F.col("is_fraud") == 1).count()
nonfraud_count = train_df.filter(F.col("is_fraud") == 0).count()

num_classes = 2

weight_0 = total_count / (num_classes * nonfraud_count)
weight_1 = total_count / (num_classes * fraud_count)

train_weighted = train_df.withColumn(
    "class_weight",
    F.when(F.col("is_fraud") == 1, weight_1).otherwise(weight_0)
)

test_weighted = test_df.withColumn("class_weight", F.lit(1.0))

In [48]:
lr_w_pipeline = Pipeline(stages=indexers + encoders + [assembler, lr_w])

weighted_preds, weighted_summary = run_experiment(
    lr_w_pipeline,
    train_weighted,
    test_weighted,
    model_name="Weighted Logistic Regression with Feature Engineering",
    label_col="is_fraud",
    pred_col="prediction"
)

build_comparison_df(spark, [weighted_summary]).show(truncate=False)

+-----------------------------------------------------+------+-----+---+----+---------------+------------+--------+---------------+
|model                                                |tn    |fp   |fn |tp  |fraud_precision|fraud_recall|fraud_f1|nonfraud_recall|
+-----------------------------------------------------+------+-----+---+----+---------------+------------+--------+---------------+
|Weighted Logistic Regression with Feature Engineering|220672|37664|336|1236|0.0318         |0.7863      |0.0611  |0.8542         |
+-----------------------------------------------------+------+-----+---+----+---------------+------------+--------+---------------+



## Model Tuning

After establishing baseline performance, correcting class imbalance, and introducing feature engineering, the next step is model tuning.

This section evaluates both Logistic Regression and Random Forest using the same train, validation, and test framework:

- the **training** set is used to fit each model
- the **validation** set is used to compare parameter settings and thresholds
- the **test** set is reserved for final evaluation only

Logistic Regression is tuned first because it is faster to iterate on and easier to interpret. The insights from that process are then used to guide Random Forest tuning, which is more flexible but also more computationally expensive.

All tuning decisions are evaluated using fraud-focused metrics so that both models can be compared consistently on the actual objective of fraud detection.

In [49]:
# Create train, test, and validate datasets
train_base_df, test_df = model_df.randomSplit([0.8, 0.2], seed=5420)
train_df, val_df = train_base_df.randomSplit([0.8, 0.2], seed=5420)

val_weighted = val_df.withColumn("class_weight", F.lit(1.0))
test_weighted = test_df.withColumn("class_weight", F.lit(1.0))

## Logistic Regression

Logistic Regression is tuned first to establish a strong and interpretable benchmark. The goal is to determine which parameters meaningfully improve fraud detection performance before moving to Random Forest.

### Running the Logistic Regression Parameter Sweep

Each parameter combination is fit on the training set and evaluated on the validation set. This makes it possible to compare how regularization and convergence settings affect fraud precision, fraud recall, and fraud F1.

In [50]:
reg_params = [0.01, 0.1, 0.5]
elastic_params = [0.0, 0.25, 0.5, 0.75, 1.0]
max_iters = [50, 100]

In [51]:
param_results = []

for reg in reg_params:
    for elastic in elastic_params:
        for max_iter in max_iters:
            lr_tuned = LogisticRegression(
                featuresCol="features",
                labelCol=label_col,
                weightCol="class_weight",
                predictionCol="prediction",
                probabilityCol="probability",
                rawPredictionCol="rawPrediction",
                maxIter=max_iter,
                regParam=reg,
                elasticNetParam=elastic
            )

            lr_tuned_pipeline = Pipeline(
                stages=indexers + encoders + [assembler, lr_tuned]
            )

            preds, summary = run_experiment(
                lr_tuned_pipeline,
                train_weighted,
                val_weighted,
                model_name=f"reg={reg}, elastic={elastic}, maxIter={max_iter}",
                label_col="is_fraud",
                pred_col="prediction"
            )

            param_results.append(summary)

### Logistic Regression Tuning Results

In [52]:
param_results_df = build_comparison_df(spark, param_results)

param_results_df.orderBy(
    F.desc("fraud_f1"),
    F.desc("fraud_recall")
).show(10, truncate=False)

+----------------------------------+------+-----+---+---+---------------+------------+--------+---------------+
|model                             |tn    |fp   |fn |tp |fraud_precision|fraud_recall|fraud_f1|nonfraud_recall|
+----------------------------------+------+-----+---+---+---------------+------------+--------+---------------+
|reg=0.5, elastic=0.25, maxIter=50 |191253|15463|286|922|0.0563         |0.7632      |0.1048  |0.9252         |
|reg=0.5, elastic=0.25, maxIter=100|191253|15463|286|922|0.0563         |0.7632      |0.1048  |0.9252         |
|reg=0.1, elastic=1.0, maxIter=50  |190289|16427|286|922|0.0531         |0.7632      |0.0994  |0.9205         |
|reg=0.1, elastic=1.0, maxIter=100 |190289|16427|286|922|0.0531         |0.7632      |0.0994  |0.9205         |
|reg=0.1, elastic=0.75, maxIter=50 |190377|16339|291|917|0.0531         |0.7591      |0.0993  |0.921          |
|reg=0.1, elastic=0.75, maxIter=100|190377|16339|291|917|0.0531         |0.7591      |0.0993  |0.921    

### Interpretation of Logistic Regression Tuning

The Logistic Regression tuning results show that `maxIter` has no measurable impact on performance. The same parameter combinations produce identical results at 50, and 100 iterations, indicating that the model is already converging efficiently at lower values.

Regularization has a much larger effect. The models with little or no regularization perform worse, while moderate regularization produces clear gains in fraud-focused performance. 

The strongest Logistic Regression configuration is:

- `regParam = 0.5`
- `elasticNetParam = 0.25`
- `maxIter = 50`

Resulting in scores of:

- Fraud Recall: 76.32%
- Fraud Precision: 5.63%
- Fraud F1: 10.48%

Compared to the baseline Logistic Regression model, this is a major improvement. The baseline model identified only about 1.3% of fraudulent transactions, while this tuned configuration captures over 76% of fraud cases.

This suggests that the weighting and tuning process substantially improved the model’s sensitivity to fraud and helped address the class imbalance problem. However, precision remains low, meaning the model still produces a large number of false positives.

Overall, this configuration appears to be the strongest Logistic Regression result, and will be used as the starting point for threshold tuning.

### Final Logistic Regression Model

Based on the validation results, the strongest Logistic Regression configuration is selected for the next stage of evaluation. This model will be used for probability scoring and threshold tuning.

In [53]:
best_lr = LogisticRegression(
    featuresCol="features",
    labelCol=label_col,
    weightCol="class_weight",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxIter=50,
    regParam=0.5,
    elasticNetParam=0.25
)

best_pipeline = Pipeline(stages=indexers + encoders + [assembler, best_lr])

best_preds, _ = run_experiment(
    best_pipeline,
    train_weighted,
    val_weighted,
    model_name="Best tuned LR",
    label_col="is_fraud",
    pred_col="prediction"
)

scored_preds = best_preds.withColumn(
    "fraud_prob",
    vector_to_array("probability")[1]
)

### Threshold Tuning for Logistic Regression

After selecting the strongest Logistic Regression configuration, the next step is to tune the decision threshold.

The model produces probabilities, but the default threshold is not necessarily the best operating point for fraud detection. Lower thresholds make the model more aggressive and increase fraud recall, while higher thresholds reduce false positives but miss more fraud.

Threshold tuning is evaluated on the validation set so that the final test set remains untouched until a final decision is made.

In [54]:
def evaluate_threshold(df, threshold):
    temp = df.withColumn(
        "custom_prediction",
        F.when(F.col("fraud_prob") >= threshold, 1).otherwise(0)
    )
    
    return summarize_binary_predictions(
        temp,
        label_col="is_fraud",
        pred_col="custom_prediction",
        model_name=f"Threshold {threshold}"
    )

In [55]:
#threshold values above 0.7 and below 0.4 had diminishing retunrs in testing and were removed from evaluations
thresholds = [0.7 ,0.6, 0.55, 0.525, 0.5125, 0.5, 0.4]
threshold_results = [evaluate_threshold(scored_preds, t) for t in thresholds]

### Logistic Regression Threshold Results

In [56]:
threshold_df = spark.createDataFrame(threshold_results)
build_comparison_df(spark, threshold_results).show()

+----------------+------+------+----+----+---------------+------------+--------+---------------+
|           model|    tn|    fp|  fn|  tp|fraud_precision|fraud_recall|fraud_f1|nonfraud_recall|
+----------------+------+------+----+----+---------------+------------+--------+---------------+
|   Threshold 0.7|206421|   295|1200|   8|         0.0264|      0.0066|  0.0106|         0.9986|
|   Threshold 0.6|205637|  1079| 727| 481|         0.3083|      0.3982|  0.3475|         0.9948|
|  Threshold 0.55|203517|  3199| 599| 609|         0.1599|      0.5041|  0.2428|         0.9845|
| Threshold 0.525|201334|  5382| 350| 858|         0.1375|      0.7103|  0.2304|          0.974|
|Threshold 0.5125|198492|  8224| 316| 892|         0.0978|      0.7384|  0.1728|         0.9602|
|   Threshold 0.5|191253| 15463| 286| 922|         0.0563|      0.7632|  0.1048|         0.9252|
|   Threshold 0.4| 42324|164392|  29|1179|         0.0071|       0.976|  0.0141|         0.2047|
+----------------+------+-----

### Interpretation of Threshold Tuning

The threshold tuning results show a clear tradeoff between fraud recall and fraud precision.

As the threshold is lowered, the model becomes more aggressive in labeling transactions as fraud. This substantially increases fraud recall, but it also sharply increases false positives and reduces precision.

In these results, that tradeoff is very clear:

- At a threshold of `0.7`, fraud precision is relatively higher at **2.64%**, but fraud recall falls to only **0.66%**, meaning almost all fraud is missed.
- At the default threshold of `0.5`, fraud recall improves to **76.32%**, but precision drops to **5.63%**.
- At `0.4`, fraud recall rises to **97.60%**, but precision collapses to **0.71%**, with **164,392 false positives**, making that threshold operationally unrealistic.

These results show that threshold tuning meaningfully improves fraud capture, but maximizing recall alone is not sufficient. Extremely low thresholds create too many false positives to be practical in a real fraud detection setting.

Based on the validation results, the most reasonable operating region appears to be around `0.525` to `0.5`, where fraud recall improves substantially while still avoiding the extreme false-positive burden seen at `0.4`.

The final threshold value of `0.525` was selected based on the balance between fraud capture and operational cost, rather than on recall alone. At this threshold, the model maintains strong fraud recall while avoiding the extreme increase in false positives observed at lower thresholds.

### Logistic Regression Summary

The tuned Logistic Regression model represents a major improvement over the baseline model. Through regularization tuning and threshold adjustment, the model moved from identifying almost no fraudulent transactions to capturing a substantial share of fraud cases.

The final Logistic Regression configuration uses:

- `regParam = 0.5`
- `elasticNetParam = 0.25`
- `maxIter = 50`
- `threshold = 0.525`

This combination was selected because it provides a stronger balance between fraud detection and operational cost than the default threshold or more extreme threshold values. In particular, it improves fraud recall substantially while avoiding the unsustainably high false-positive burden observed at lower thresholds.

Even so, Logistic Regression remains a relatively simple linear model. While it performs much better after tuning, its low fraud precision suggests that it may still struggle to separate complex fraud patterns from legitimate transactions.

To determine whether a more flexible model can improve this tradeoff, the next step is to evaluate Random Forest, which can capture nonlinear relationships and feature interactions that Logistic Regression may miss.

## Random Forest Model

After improving Logistic Regression through regularization and threshold tuning, the next step is to evaluate Random Forest as a more flexible model.

Unlike Logistic Regression, Random Forest can capture nonlinear relationships and interactions between features. This is especially valuable in fraud detection, where fraudulent behavior may depend on complex combinations of variables rather than simple linear patterns.

The purpose of this section is to determine whether Random Forest can improve fraud detection performance, especially by increasing fraud recall and precision relative to Logistic Regression.

### Parameter Setup

Random Forest is tuned using the same train, validation, and test framework as Logistic Regression. This keeps the comparison consistent and allows performance differences to be attributed to the model itself rather than to changes in data preparation.

Random Forest is a useful next step because it can capture nonlinear relationships and feature interactions, which are common in fraudulent transaction patterns.

#### Selected Hyperparameters

The first round of Random Forest tuning focuses on `numTrees`, `minInstancesPerNode`, and `featureSubsetStrategy`.

These parameters are evaluated first because they are less computationally expensive than depth tuning and help establish a strong baseline configuration. Once the best-performing combination is identified, `maxDepth` can then be tuned separately to test whether additional model complexity improves fraud detection performance.

In [57]:
num_trees_list = [20, 50, 100]
min_instances_list = [1, 2, 5, 10]
feature_subset_list = ["sqrt", "log2", "onethird"]

Note: Since Random Forest algorithms can be expensive, we cache the datasets in order to free up some resources for Spark.

In [58]:
train_weighted = train_weighted.cache()
val_weighted = val_weighted.cache()

train_weighted.count()
val_weighted.count()

rf_results = []

for n_trees in num_trees_list:
    for min_inst in min_instances_list:
        for feat_subset in feature_subset_list:

            rf_model = RandomForestClassifier(
                featuresCol="features",
                labelCol=label_col,
                weightCol="class_weight",
                predictionCol="prediction",
                probabilityCol="probability",
                rawPredictionCol="rawPrediction",
                numTrees=n_trees,
                minInstancesPerNode=min_inst,
                featureSubsetStrategy=feat_subset,
                seed=5420
            )

            rf_tune_pipeline = Pipeline(
                stages=indexers + encoders + [assembler, rf_model]
            )

            preds, summary = run_experiment(
                rf_tune_pipeline,
                train_weighted,
                val_weighted,
                model_name=f"trees={n_trees}, minNode={min_inst}, feat={feat_subset}",
                label_col="is_fraud",
                pred_col="prediction"
            )

            rf_results.append(summary)

In [59]:
rf_results_df = build_comparison_df(spark, rf_results)

rf_results_df.orderBy(
    F.desc("fraud_recall"),
    F.desc("fraud_f1"),
    F.desc("fraud_precision")
).show(truncate=False)

+------------------------------------+------+----+---+----+---------------+------------+--------+---------------+
|model                               |tn    |fp  |fn |tp  |fraud_precision|fraud_recall|fraud_f1|nonfraud_recall|
+------------------------------------+------+----+---+----+---------------+------------+--------+---------------+
|trees=20, minNode=5, feat=onethird  |197398|9318|102|1106|0.1061         |0.9156      |0.1902  |0.9549         |
|trees=20, minNode=10, feat=onethird |198392|8324|114|1094|0.1162         |0.9056      |0.2059  |0.9597         |
|trees=100, minNode=2, feat=onethird |199003|7713|118|1090|0.1238         |0.9023      |0.2178  |0.9627         |
|trees=100, minNode=1, feat=onethird |199003|7713|118|1090|0.1238         |0.9023      |0.2178  |0.9627         |
|trees=50, minNode=5, feat=onethird  |199920|6796|119|1089|0.1381         |0.9015      |0.2395  |0.9671         |
|trees=100, minNode=5, feat=onethird |198596|8120|119|1089|0.1183         |0.9015      |

#### Interpretation of Random Forest Tuning

The Random Forest tuning results show that `featureSubsetStrategy = onethird` consistently performs best among the tested configurations. This is reasonable, since using a subset of available features at each split helps increase tree diversity while still preserving useful predictive patterns.

The effects of `numTrees` and `minInstancesPerNode` are more mixed. While larger forests often appear among the strongest configurations, smaller values can still perform competitively, suggesting that adding more trees does not always produce a large improvement relative to the additional computational cost.

Because this project is focused on fraud detection, the strongest configuration should be judged using fraud recall, fraud precision, and fraud F1 rather than overall weighted metrics. Based on the validation results, the selected configuration provides the best balance between capturing fraud and limiting false positives among the tested combinations.

The strongest Random Forest configuration is:

- `numTees = 50`
- `minInstancesPerNode = 5`
- `featureSubsetStrategy = onethird`

Resulting in scores of:

- Fraud Recall: 90.15%
- Fraud Precision: 13.81%
- Fraud F1: 23.95%

Compared to the baseline Random Forest model, this is a major improvement. The baseline model identified only about 26% of fraudulent transactions, while this tuned configuration captures over 90% of fraud cases.

This suggests that the weighting and tuning process substantially improved the model’s sensitivity to fraud and helped address the class imbalance problem. However, precision remains low, meaning the model still produces a large number of false positives.

Overall, this configuration appears to be the strongest Random Forest result, and will be used as the starting point for depth tuning.

### Depth Tuning for Random Forest

After identifying the strongest Random Forest configuration from the initial tuning step, the next stage is to evaluate `maxDepth`.

Tree depth controls model complexity. Shallower trees are simpler and faster to train, while deeper trees can capture more complex fraud patterns. However, increasing depth also raises computational cost and may eventually lead to diminishing returns or overfitting.

Depth tuning is evaluated on the validation set so that the final test set remains untouched until the model selection process is complete.

In [60]:
# Best Hyperparameters

b_numTree = 50
b_minInst = 5
b_featSubset = 'onethird'

In [61]:
max_depth_list = [6, 8, 10, 12, 14]

In [62]:
train_weighted = train_weighted.cache()
val_weighted = val_weighted.cache()

train_weighted.count()
val_weighted.count()

best_rf_results = []

for depth in max_depth_list:

    best_rf_model = RandomForestClassifier(
        featuresCol="features",
        labelCol=label_col,
        weightCol="class_weight",
        predictionCol="prediction",
        probabilityCol="probability",
        rawPredictionCol="rawPrediction",
        numTrees=b_numTree,
        maxDepth=depth,
        minInstancesPerNode=b_minInst,
        featureSubsetStrategy=b_featSubset,
        seed=5420
    )

    best_rf_tune_pipeline = Pipeline(
        stages=indexers + encoders + [assembler, best_rf_model]
    )

    preds, summary = run_experiment(
        best_rf_tune_pipeline,
        train_weighted,
        val_weighted,
        model_name=f"depth={depth}",
        label_col="is_fraud",
        pred_col="prediction"
    )

    best_rf_results.append(summary)

### Random Forest Depth Results

In [63]:
best_rf_results_df = build_comparison_df(spark, best_rf_results)

best_rf_results_df.orderBy(
    F.desc("fraud_recall"),
    F.desc("fraud_f1"),
    F.desc("fraud_precision")
).show(truncate=False)

+--------+------+----+---+----+---------------+------------+--------+---------------+
|model   |tn    |fp  |fn |tp  |fraud_precision|fraud_recall|fraud_f1|nonfraud_recall|
+--------+------+----+---+----+---------------+------------+--------+---------------+
|depth=14|204548|2168|25 |1183|0.353          |0.9793      |0.519   |0.9895         |
|depth=12|203439|3277|35 |1173|0.2636         |0.971       |0.4146  |0.9841         |
|depth=10|202357|4359|40 |1168|0.2113         |0.9669      |0.3468  |0.9789         |
|depth=8 |200870|5846|56 |1152|0.1646         |0.9536      |0.2808  |0.9717         |
|depth=6 |200052|6664|76 |1132|0.1452         |0.9371      |0.2514  |0.9678         |
+--------+------+----+---+----+---------------+------------+--------+---------------+



### Interpretation of Depth Tuning

The depth results show a tradeoff between model complexity and performance:

- Shallow Depths increase recall by identifying more fraudulent transactions
- Deep Depths improve precision by reducing false positives

Because the purpose of this project is fraud detection, recall is especially important. However, the depth still needs to remain operationally practical so that the number of false positives does not become unmanageable.

The final depth will therefore be chosen based on the balance between fraud capture and operational cost, rather than on a single metric alone.

## Model Comparison (Logistic Regression vs Random Forest)

At this stage, both models demonstrate meaningful fraud detection capability, but with different strengths.

- Logistic Regression achieves strong fraud recall (~76%) and provides a simple, highly interpretable model. However, it suffers from very low precision (~5.6%), resulting in a high number of false positives.

- Random Forest significantly improves overall performance, achieving both higher fraud recall (~98%) and substantially higher precision (~35%). This indicates a much stronger ability to distinguish between fraudulent and legitimate transactions.

These results suggest that Random Forest is better suited for maximizing fraud detection performance, while Logistic Regression remains valuable for its simplicity and interpretability.

Both models are viable candidates for production use, depending on the specific operational priorities. Logistic Regression may be preferred in environments where transparency and simplicity are critical, while Random Forest is better suited for scenarios where maximizing fraud detection accuracy is the primary objective.

For this reason, both models will be retained in the final codebase to support different use cases and evaluation scenarios.

## Saving Model Outputs for Visualization

The results below are saved to disk to enable visualization and further analysis. 

By saving these datasets, visualizations can be created independently of the modeling process without requiring the models to be retrained.

In [68]:
models_path = repo_root / "data" / "models"

param_results_df.write.mode("overwrite").parquet(str(models_path / "lr_param_tuning"))
threshold_df.write.mode("overwrite").parquet(str(models_path / "lr_threshold_tuning"))
scored_preds.select("is_fraud", "prediction", "fraud_prob").write.mode("overwrite").parquet(str(models_path / "lr_scored_predictions"))
rf_results_df.write.mode("overwrite").parquet(str(models_path / "rf_param_tuning"))
best_rf_results_df.write.mode("overwrite").parquet(str(models_path / "rf_depth_tuning"))